# 📘 Customer Churn & CLTV Prediction — Full Data Science Notebook
This notebook includes:
- Data Loading
- Full EDA
- Feature Engineering
- Churn Model
- CLTV Model
- Visualizations
- Final Insights


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, roc_auc_score, mean_squared_error

sns.set(style='whitegrid')
pd.set_option('display.max_columns', None)
print('Libraries Loaded Successfully!')

## 📂 Load Input Files

In [ ]:
customers = pd.read_csv('../inputs/customers.csv', parse_dates=['signup_date'])
transactions = pd.read_csv('../inputs/transactions.csv', parse_dates=['transaction_date'])
events = pd.read_csv('../inputs/events.csv', parse_dates=['event_date'])
support = pd.read_csv('../inputs/support.csv', parse_dates=['ticket_date'])

customers.head()

## 📏 Dataset Shapes

In [ ]:
print('Customers:', customers.shape)
print('Transactions:', transactions.shape)
print('Events:', events.shape)
print('Support:', support.shape)

## ❗ Missing Values

In [ ]:
for name, df in [('Customers', customers),
                 ('Transactions', transactions),
                 ('Events', events),
                 ('Support', support)]:
    print(f'--- {name} ---')
    print(df.isnull().sum(), '\n')

## 📊 Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=customers, x='plan')
plt.title('Plan Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=customers, x='churn')
plt.title('Churn Distribution')
plt.show()

## 🧮 Transaction-Based Features

In [ ]:
trans_agg = transactions.groupby('customer_id').agg(
    total_spend=('amount','sum'),
    trans_count=('amount','count'),
    last_transaction=('transaction_date','max')
).reset_index()
trans_agg.head()

## 🔐 Engagement Features (Logins)

In [ ]:
ref_date = customers['signup_date'].max() + pd.Timedelta(days=365)
events['days_from_ref'] = (ref_date - events['event_date']).dt.days

login_7 = events[events['days_from_ref'] <= 7].groupby('customer_id').size().rename('logins_7d')
login_30 = events[events['days_from_ref'] <= 30].groupby('customer_id').size().rename('logins_30d')
login_90 = events[events['days_from_ref'] <= 90].groupby('customer_id').size().rename('logins_90d')

logins = pd.concat([login_7, login_30, login_90], axis=1).fillna(0).reset_index()
logins.head()

## 🛠 Support Features

In [ ]:
support_agg = support.groupby('customer_id').agg(
    tickets=('issue_type','count'),
    avg_resolution=('resolution_days','mean')
).reset_index()
support_agg.head()

## 🔄 Merge All Features

In [ ]:
df = customers.merge(trans_agg, on='customer_id', how='left')
df = df.merge(logins, on='customer_id', how='left')
df = df.merge(support_agg, on='customer_id', how='left')
df = df.fillna(0)
df.head()

## 🔥 Correlation Analysis

In [ ]:
df_corr = df.copy()
df_corr['plan_enc'] = LabelEncoder().fit_transform(df_corr['plan'])
plt.figure(figsize=(12,6))
sns.heatmap(df_corr.corr(), cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

# 🤖 Churn Model Development

In [ ]:
le = LabelEncoder()
df['plan_enc'] = le.fit_transform(df['plan'])

features = [
    'monthly_fee','recency_days','total_spend','trans_count',
    'logins_7d','logins_30d','logins_90d','tickets','avg_resolution','plan_enc'
]

X = df[features]
y = df['churn'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

churn_model = RandomForestClassifier(n_estimators=300, random_state=42)
churn_model.fit(X_train, y_train)

y_pred = churn_model.predict(X_test)
y_prob = churn_model.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred))
print('ROC-AUC:', roc_auc_score(y_test, y_prob))

## 🔍 Churn Model Feature Importance

In [ ]:
imp = pd.Series(churn_model.feature_importances_, index=features)
imp.sort_values().plot(kind='barh', figsize=(8,6))
plt.title('Feature Importance (Churn Model)')
plt.show()

# 💰 CLTV Model Development

In [ ]:
df['cltv_label'] = df['total_spend'] * 1.5
X2 = df[features]
y2 = df['cltv_label']

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.25, random_state=42)

cltv_model = RandomForestRegressor(n_estimators=300, random_state=42)
cltv_model.fit(X2_train, y2_train)

preds = cltv_model.predict(X2_test)
rmse = mean_squared_error(y2_test, preds, squared=False)

print('CLTV RMSE:', rmse)

## 💾 Save Models

In [ ]:
joblib.dump(churn_model, '../models/churn_model_notebook.pkl')
joblib.dump(cltv_model, '../models/cltv_model_notebook.pkl')
print('Models Saved Successfully')